#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim,col
from pyspark.sql.window import Window

In [0]:
rename_map = {
    'prd_id':'product_id',
    'prd_key':'product_key',
    'product_category_id':'product_category_id',
    'prd_nm':'product_name',
    'prd_cost':'product_cost',
    'prd_line':'product_line',
    'prd_start_dt':'product_start_date',
    'prd_end_dt':'product_end_date',}

#Reading Data From Bronze Layer

In [0]:
df = spark.table('workspace.bronze.crm_prd_info')

#Data Cleaning

##Column Splitting

In [0]:
df = df.withColumn('product_category_id', F.substring(F.trim(col('prd_key')), 1, 5))
df = df.withColumn('prd_key', F.substring(F.trim(col('prd_key')), 7, F.length(('prd_key'))))

##Trimming String columns

In [0]:
for column in df.dtypes:
    if column[1] == 'string':
        df = df.withColumn(column[0],trim(col(column[0])))

##Handling Null

In [0]:
df = df.withColumn('prd_cost',F.coalesce(df['prd_cost'],F.lit(0)))

##Handling Invalid date ranges

In [0]:
df = df.withColumn('prd_end_dt', F.lead('prd_start_dt', 1).over(Window.partitionBy('prd_key').orderBy('prd_start_dt'))-1)


##Standardization

###Value Mapping (e.g., R → Road)

In [0]:
df = df.withColumn(
    'prd_line',
    F.when(F.upper(F.trim('prd_line')) == 'R', 'Road')
    .when(F.upper(F.trim('prd_line')) == 'M', 'Mountain')
    .when(F.upper(F.trim('prd_line')) == 'T', 'Touring')
    .when(F.upper(F.trim('prd_line')) == 'S', 'Other Sales')
    .otherwise('Unknown')
)

###Column Naming Standardization

In [0]:
for old_name,new_name in rename_map.items():
    df = df.withColumnRenamed(old_name,new_name)


#Loading Data Into Silver Layer

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('silver.crm_product_info')